#### **문제**
- GridSearchCV와 연동하기 위해서 Word2VecVectorizer Class를 보강

---

- 모델을 선택할 수 있도록 **생성자 함수에 추가적인 작업**
    - 4개의 매개변수 추가
        - `min_n` (default 2), `max_n` (default 6), `bucket` (default 2,000,000)
        - model을 선택할 수 있는 매개변수 추가: `type` (default `w2v`)

- **fit** 함수 수정
    - `self.type` 에 따라서 학습되는 모델을 변경
        - `w2v` 라면 → Word2Vec 학습하여 `self.model`에 대입
        - `ft` 라면 → FastText 학습하여 `self.model`에 대입

- 해당 클래스를 **모듈화**
    - 모듈 이름: `gensim_test`

---

1. 모듈 로드
2. tokenizer는 Okt 사용
    - okt.morphs() 사용하여 토큰화
3. `ratings_test.txt` 데이터 로드
4. 결측치 제거
5. 글자 간의 좌우 공백을 제거
6. 빈 테스트 데이터가 document에 존재하는가? 존재한다면 제외
7. 중복되는 데이터를 제외
8. 상위 데이터 100개를 이용, GridSearchCV를 사용하여 parameter 조합
    - parameter 조합 (vectorize: **주의! min_count = 1**)
        - type: ['w2v', 'ft']
        - vector_size: [80, 100]
    - parameter 조합 (svc)
        - C: [0.8, 1.0]
    - 계층화 폴드는 5회
9. 하위 데이터 100개를 이용하여 검증 : 분류 레포트를 이용

In [72]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.svm import SVC

from sklearn.metrics import classification_report
from konlpy.tag import Okt
from gensim.models import Word2Vec, FastText

##### 모듈 수정

In [ ]:
# # Class 선언: Word2Vec을 GridSearchCV에 사용하기 위해서

# class Word2VecVectorizer(BaseEstimator, TransformerMixin):
#     # BaseEstimator: get_params, set_params와 같은 함수의 기능을 상속 받는다.
#     # TransformerMixin: fit()과 transform() 함수만 선언하면 fit_transform() 함수를 이용 가능


#     # 생성자 함수 → class가 생성될 때 기본적으로 사용할 변수들을 지정 (데이터 대입)
#         # Word2Vec에서 사용할 인자값들을 생성자 함수에서 미리 받아온다.
#     def __init__(
#             self,                   # 자기 자신: 객체가 생성된 위치
#             tokenizer = None,       # 토큰화 함수 (기본값은 None)
#             vector_size = 100,      # 벡터화된 데이터의 차원의 수 지정
#             min_count = 5,          # 전체 문서에서 최소 등장 횟수 지정
#             window = 5,             # 중심 단어와 주변 단어들의 거리 제한
#             sg = 1,                 # 단어 예측 방식 (0: CBOW, 1: skip-gram)
#             epochs = 100,           # 반복 학습의 횟수
#             workers = 1,            # 계산에 사용할 스레드의 수
#             seed = 42,
#             min_n = 2,
#             max_n = 6,
#             bucket = 2000000
#             type = 'w2v'
#             l2 = False
#     ):
#         self.tokenizer = tokenizer
#         self.vector_size = vector_size
#         self.min_count = min_count
#         self.window = window
#         self.sg = sg
#         self.epochs = epochs
#         self.workers = workers
#         self.seed = seed
#         self.min_n = min_n
#         self.max_n = max_n
#         self.bucket = bucket
#         self.type = type
#         self.l2 = l2
    
#         # 모델과 단어 사전을 저장할 빈 공간 생성
#             # 일반적인 문법: class 선언 시 self.변수(객체 변수)들은 생성자 함수에서 생성한다.
#         self.model = None
#         self.voca = None
    



#     # 총 4개의 메서드를 생성: 토큰화, 학습, 문장 데이터를 평균 단위 벡터로 생성하는 함수, 변형
#     # 토큰화 메서드
#     def to_token(self, sentences):
#         # sentences: 문장들의 목록
#         # 만약에 토큰화 함수가 존재하지 않는다면: self.tokenizer가 None인 경우 → split()
#         if self.tokenizer is None:
#             result = []
#             for sentence in sentences:
#                 token = list(sentence.split())
#                 result.append(token)
#             # result = [[word for word in sentence.split()] for sentence in sentences]
#         else:
#             result = []
#             for sentence in sentences:
#                 token = self.tokenizer(sentence)
#                 result.append(token)
#             # result = [self.tokenizer(sentence) for sentence in sentences]
#         return result
    

#     # 학습 메서드: fit() 함수 생성
#     # sklearn 안 모델들의 fit() 함수의 인자값들: 독립변수, 종속변수
#     def fit(self, X, y):
#         # X: 독립 변수 (문장 목록, 2차원 데이터)
#         # y: 종속 변수 (1차원 데이터)
#         # sentences: 토큰화된 문장 데이터
#         sentences = self.to_token(X)
#         # self.type이 Word2Vec이라면 Word2Vec에 학습
#         if self.type == 'w2v':
#             self.model = Word2Vec(
#                 sentences = sentences,
#                 vector_size = self.vector_size,
#                 window = self.window,
#                 min_count = self.min_count,
#                 sg = self.sg,
#                 epochs = self.epochs,
#                 workers = self.workers,
#                 seed = self.seed
#             )
        
#         elif self.type == 'ft':
#             self.model = FastText(
#                 sentences = sentences,
#                 vector_size = self.vector_size,
#                 window = self.window,
#                 min_n = self.min_n,
#                 max_n = self.max_n,
#                 bucket = self.bucket,
#                 sg = self.sg,
#                 epochs = self.epochs,
#                 workers = self.workers,
#                 seed = self.seed
#             )

#         # 학습 모델이 생성되었으니 단어 사전에 데이터 입력
#         self.voca = self.model.wv.key_to_index.keys()
#         return self
    

#     # 문장 벡터 생성하는 함수
#     def doc_vec(self, token):
#         # token: 토큰화된 문장 데이터 (1개의 문장)
#         vectors = []
#         for word in token:
#             if word in self.model.wv:
#                 vec = self.model.wv[word]
#                 vectors.append(vec)
#         # vectors 데이터가 존재하지 않는 경우 → 특정 문장에서 단어들이 단어 사전에 존재하지 않을 때
#         if vectors:
#             result = np.mean(vectors, axis = 0)
#         else:
#             result = np.zeros(self.model.vector_size)
#         return result
    

#     # 변형 함수 생성 (transform)
#     # sklearn 안 모델들의 transform() 함수의 인자값: X_test
#     def transform(self, X):
#         # 토큰화
#         sentences = self.to_token(X)
#         # 벡터화(임베딩)
#         result = []
#         for token in sentences:
#             vec = self.doc_vec(token)
#             result.append(vec)
#         return np.array(result)
    

##### 돌아와서

In [74]:
import gensim_test

In [75]:
okt = Okt()

In [104]:
df = pd.read_csv('../data/ratings_test.txt', sep = '\t')

In [105]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        50000 non-null  int64
 1   document  49997 non-null  str  
 2   label     50000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.1 MB


In [106]:
df.drop('id', axis = 1, inplace = True)

In [107]:
df.dropna(inplace = True)
df.reset_index(drop=True, inplace=True)
df

,document,label
0,굳 ㅋ,1
1,GDNTOPCLASSINTHECLUB,0
2,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0
...,...,...
49992,오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함,1
49993,의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO,0
49994,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49995,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0


In [108]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49997 entries, 0 to 49996
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   document  49997 non-null  str  
 1   label     49997 non-null  int64
dtypes: int64(1), str(1)
memory usage: 781.3 KB


In [109]:
df.dropna(inplace = True)

In [110]:
for idx, data in enumerate(df['document']):
    df.loc[idx, 'document'] = data.strip()

In [111]:
flag = (df['document'] == '')
df.loc[flag]

,document,label


In [112]:
df['document'].value_counts()

document
굿                                                 56
good                                              35
최고                                                30
tv 전기세가 아깝다!!!                                    20
별로                                                20
                                                  ..
오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함          1
의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO       1
그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다                 1
절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네     1
마무리는 또 왜이래                                         1
Name: count, Length: 49157, dtype: int64

In [113]:
df.drop_duplicates('document', inplace = True)

In [114]:
df['document'].value_counts()

document
굳 ㅋ                                                  1
GDNTOPCLASSINTHECLUB                                 1
뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아               1
지루하지는 않은데 완전 막장임... 돈주고 보기에는....                     1
3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??    1
                                                    ..
오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함            1
의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO         1
그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다                   1
절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네       1
마무리는 또 왜이래                                           1
Name: count, Length: 49157, dtype: int64

In [115]:
tokenizer = lambda x: [ word for word in okt.morphs(x) ]

pipe = Pipeline(
    [
        ('emb', gensim_test.Word2VecVectorizer(tokenizer = tokenizer, min_count = 1)),
        ('svc', SVC())
    ]
)

In [116]:
params_grid = {
    'emb__type': ['w2v', 'ft'],
    'emb__vector_size': [80, 100],
    'svc__C': [0.8, 1.0]
}

In [117]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

In [118]:
grid = GridSearchCV(
    estimator = pipe,
    param_grid = params_grid,
    cv = cv,
    verbose = 1
)

In [119]:
X = df['document'][:100].values
y = df['label'][:100].values

In [120]:
grid.fit(X, y)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...svc', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'emb__type': ['w2v', 'ft'], 'emb__vector_size': [80, 100], 'svc__C': [0.8, 1.0]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_

In [121]:
print('GridSearchCV 기준 최적의 파라미터:', grid.best_params_)
print('GridSearchCV 최적의 점수:', grid.best_score_)

GridSearchCV 기준 최적의 파라미터: {'emb__type': 'w2v', 'emb__vector_size': 100, 'svc__C': 0.8}
GridSearchCV 최적의 점수: 0.55


In [127]:
df2 = df.tail(100)
df2.dropna(inplace=True)
X_test = df2['document'].values
y_test = df2['label'].values

In [128]:
pred = grid.predict(X_test)
pred

array([0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1,
       0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [129]:
print(np.isnan(pred).any())

False


In [130]:
y_test

array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0])

In [131]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.60      0.78      0.68        51
           1       0.67      0.45      0.54        49

    accuracy                           0.62       100
   macro avg       0.63      0.62      0.61       100
weighted avg       0.63      0.62      0.61       100

